In [1]:
# import Required Libraries
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

In [2]:
# Set random seeds for reproducibility
torch.manual_seed(42)

In [3]:
# Check GPU availability
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Available Device is: {device}')

Available Device is: cuda


In [4]:
df = pd.read_csv('/content/fashion-mnist_train.csv')
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,0,1,2,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [5]:
# Extract Features values and Target values
X=df.iloc[:, 1:].to_numpy()
y=df.iloc[:, 0].to_numpy()

In [6]:
# Split the Dataset into train and testset
X_train,X_test, y_train,y_test=train_test_split(
    X,y, test_size=0.2, random_state=42
)

In [7]:
# Scaling the features
X_train=X_train/255.0
X_test=X_test/255.0

In [25]:
# Create CustomDataset Class
class CustomDataset(Dataset):

    def __init__(self, features, labels):
        self.features=torch.tensor(features, dtype=torch.float32).reshape(-1,1,28,28)
        self.labels=torch.tensor(labels, dtype=torch.long)
    def __len__(self):
        return len(self.features)
    def __getitem__(self, index):
        return self.features[index], self.labels[index]

In [26]:
# Create train dataset object
train_dataset=CustomDataset(X_train, y_train)

In [27]:
# Create test Dataset object
test_dataset=CustomDataset(X_test, y_test)

In [28]:
# Load Dataset
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, pin_memory=True)

In [29]:
from torch.nn.modules import Conv2d
# Define Neural Network Class

class MyNN(nn.Module):

    def __init__(self, input_features):
        super().__init__()

        # Creating Convolutional Layer
        self.features=nn.Sequential(
            # 1st Conv Layer
            nn.Conv2d(input_features, 32, kernel_size=3, padding='same'),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # 2nd Conv Layer
            nn.Conv2d(32, 64, kernel_size=3, padding='same'),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
            )

        # Creating Classification Layer
        self.classifier=nn.Sequential(
            nn.Flatten(),
            # 1st layer
            nn.Linear(64*7*7, 128),     # Input came from conv layer
            nn.ReLU(),
            nn.Dropout(p=0.4),

            # 2nd Layer
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(p=0.4),

            # Output Layer
            nn.Linear(64, 10)
            )
    def forward(self, x):
        x= self.features(x)
        x= self.classifier(x)

        return (x)

In [30]:
# Define Learning Rate and Epochs
learning_rate = 0.01
epochs = 100

In [31]:
# Create model object
model = MyNN(1)

model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=learning_rate, weight_decay=1e-4)

In [32]:
# training loop

for epoch in range(epochs):

  total_epoch_loss = 0

  for batch_features, batch_labels in train_loader:

    # move data to gpu
    batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

    # forward pass
    outputs = model(batch_features)

    # calculate loss
    loss = criterion(outputs, batch_labels)

    # back pass
    optimizer.zero_grad()
    loss.backward()

    # update grads
    optimizer.step()

    total_epoch_loss = total_epoch_loss + loss.item()

  avg_loss = total_epoch_loss/len(train_loader)
  print(f'Epoch: {epoch + 1} , Loss: {avg_loss}')

Epoch: 1 , Loss: 0.7619002107481162
Epoch: 2 , Loss: 0.44850485124687356
Epoch: 3 , Loss: 0.3841661676218112
Epoch: 4 , Loss: 0.34786042118320865
Epoch: 5 , Loss: 0.32165765574574473
Epoch: 6 , Loss: 0.30335193380961817
Epoch: 7 , Loss: 0.2885354010735949
Epoch: 8 , Loss: 0.2766737019295494
Epoch: 9 , Loss: 0.2643493642403434
Epoch: 10 , Loss: 0.2552032128771146
Epoch: 11 , Loss: 0.2404401769153774
Epoch: 12 , Loss: 0.2369532527004679
Epoch: 13 , Loss: 0.2272717562870433
Epoch: 14 , Loss: 0.21953587654729684
Epoch: 15 , Loss: 0.21351240762074788
Epoch: 16 , Loss: 0.20805437974631785
Epoch: 17 , Loss: 0.20111188332767535
Epoch: 18 , Loss: 0.19664637992965678
Epoch: 19 , Loss: 0.18975745077989994
Epoch: 20 , Loss: 0.1837413517155995
Epoch: 21 , Loss: 0.18072253255546092
Epoch: 22 , Loss: 0.1702536266259849
Epoch: 23 , Loss: 0.16855366517727574
Epoch: 24 , Loss: 0.1614540524041901
Epoch: 25 , Loss: 0.157869475937138
Epoch: 26 , Loss: 0.15439182069773474
Epoch: 27 , Loss: 0.146566732670490

In [33]:
# Evaluation on test data
total = 0
correct = 0

with torch.no_grad():

  for batch_features, batch_labels in test_loader:

    # move data to gpu
    batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

    outputs = model(batch_features)

    _, predicted = torch.max(outputs, 1)

    total = total + batch_labels.shape[0]

    correct = correct + (predicted == batch_labels).sum().item()

print(correct/total)

0.9124166666666667


In [34]:
# Evaluation on training data
total = 0
correct = 0

with torch.no_grad():

  for batch_features, batch_labels in train_loader:

    # move data to gpu
    batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

    outputs = model(batch_features)

    _, predicted = torch.max(outputs, 1)

    total = total + batch_labels.shape[0]

    correct = correct + (predicted == batch_labels).sum().item()

print(correct/total)

0.9906458333333333
